ReACT agent architecture is mostly used when 2 or more queries are made to the llm in a single prompt

In [1]:
from langchain.messages import HumanMessage,AIMessage,SystemMessage
from langchain.agents import create_agent
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START,END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import add_messages
from tavily import TavilyClient
from langchain.tools import tool
from typing import TypedDict, Annotated
from langgraph.prebuilt import ToolNode, tools_condition

In [2]:
load_dotenv()

True

In [3]:
llm = init_chat_model(model="google_genai:gemini-3.6-flash")

In [4]:
tavily_client = TavilyClient()

@tool
def tavily_search(query: str) -> dict:
    """Search the web for up-to-date information, news, facts, and answers."""
    return tavily_client.search(query=query)


In [ ]:
#custom fucntion
@tool
def multiply(a:int, b:int) -> int:
    """multiply a and b
    
    args:
    a: first int
    b: second int
    
    returns:
    int: output int"""
    return a*b

In [6]:
tools = [tavily_search, multiply]
llm_with_tools = llm.bind_tools(tools)
 

In [7]:
llm_with_tools 

_ChatModelBinding(bound=ChatGoogleGenerativeAI(output_version=None, profile={}, google_api_key=SecretStr('**********'), location=None, model='gemini-3.6-flash', temperature=1.0, client=<google.genai.client.Client object at 0x0000012EB9126000>, default_metadata=(), model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'tavily_search', 'description': 'Search the web for up-to-date information, news, facts, and answers.', 'parameters': {'properties': {'query': {'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'multiply', 'description': 'multiply a and b\n\n    args:\n    a: first int\n    b: second int\n\n    returns:\n    int: output int', 'parameters': {'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [8]:
class State(TypedDict):
    messages : Annotated[list, add_messages]

In [9]:
#node functionality
def tool_calling_llm(state:State):
    #we are giving this system prompt so that the model does not loop back in aimessage and tool call which it sometimes does which burns token limit
    system = SystemMessage(content="You are a helpful assistant. Use tools when needed but avoid redundant searches. One search should be sufficient for most questions.")
    return {"messages":[llm_with_tools.invoke([system] + state['messages'])]}

In [10]:
#graph
builder =StateGraph(State)

In [11]:
#nodes
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools",ToolNode(tools))

Implementing react architecture here

In [12]:
#edges
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    #if the latest message from assistant is a tool call -> tool_condition results to tools
    #If the latest message from assitant is not a tool call -> tool_conditions routes to END"""
    tools_condition
)
builder.add_edge("tools","tool_calling_llm")

In [13]:
graph = builder.compile()

In [14]:
graph.invoke({"messages":"what is the recent news about mayor mamdani?"})

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 49.608609833s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [ ]:
graph.invoke({"messages":"WHO IS THE CURRENT MAYOR OF NEW YORK"})

{'messages': [HumanMessage(content='WHO IS THE CURRENT MAYOR OF NEW YORK', additional_kwargs={}, response_metadata={}, id='45c47b44-ac90-4359-8636-74d1169883dc'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "current Mayor of New York City"}'}, '__gemini_function_call_thought_signatures__': {'call_481299': 'Ev0CCvoCARFNMg+OROyJibvs4NL2cdi6gELfukjwakmz3MIr8gNhW+Wj8Lx7u+ScHJdSDT9f/2j6ebt8ZmUJKPHdhr6p8yNPPDlgIr4/L37/O51Zl6iF+X+AdbVKsVDmyQSE6WNhSnBqXRIIko6J0tf+RLxaRMVYERnpENAuWefD4MWkkLue4ccgDvPhlTShF0O+xUUVqY3KU5cRjP8B54rBR/caZ20eSL4W05mNsD2/FMIUWjBZLCadoVwftjbOrCSJC3bF/00HtjlOqtZun5JQHhC2+64AoeBh17FkATnWkW7iTBey9a99o9EpW0dnsvi5f9n3yhw9akYjFGR4vlwNCKxJ/RLd2v74Cx2td8j5+NapnCFAyKMoYCdr16q4YSmdNWvDE7f69cjA7mVNGPdztqM4N/FpUdpwjoTzoAWBBCk+PjHZv0B8EsKZLlCJjIrob6JP4hUe0nILtFC0Z0GrmDlrLgRWXwOZ48ePF052FosyOtIKx5U1tBJtPyyE'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_pr

In [ ]:
graph.invoke({"messages":"what is the recent news about anthropic and what is 120 multiplied by 4532"})